# Phase 3 — Leakage-Safe MiniLM Bi-Encoder Fine-Tuning

This notebook fine-tunes the pinned `all-MiniLM-L6-v2` bi-encoder on BugHub
duplicate pairs, then evaluates the trained representation with the exact same
temporal retrieval protocol as Phases 1–2.

The development protocol is deliberately two-stage:

```text
early 85% of official train (dev_fit)
    -> build historical positive pairs
    -> fine-tune from the pinned pretrained MiniLM
    -> evaluate on the latest 15% validation history
    -> compare against frozen pretrained MiniLM
    -> freeze training configuration

only if fine-tuned wins validation:
full official train
    -> rebuild pairs
    -> retrain from the original pretrained MiniLM (not the dev checkpoint)
    -> official test exactly once
```

Validation IDs and official-test IDs are forbidden from development training.
The final retrain may use the complete official train only after model selection
is frozen; official-test pairs are never used for training or negative mining.

The default notebook run builds and audits development pairs only. Training,
validation retrieval, final retraining, and official test all have separate
disabled switches.


In [ ]:
import gc
import hashlib
import html
import json
import math
import platform
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
    import sentence_transformers
    from datasets import Dataset
    from sentence_transformers import (
        SentenceTransformer,
        SentenceTransformerTrainer,
        SentenceTransformerTrainingArguments,
        losses,
    )
    from sentence_transformers.training_args import BatchSamplers
except ImportError:
    print("Installing Sentence Transformers training dependencies...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "sentence-transformers[train]",
    ])
    import torch
    import sentence_transformers
    from datasets import Dataset
    from sentence_transformers import (
        SentenceTransformer,
        SentenceTransformerTrainer,
        SentenceTransformerTrainingArguments,
        losses,
    )
    from sentence_transformers.training_args import BatchSamplers

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("CUDA available:", torch.cuda.is_available())


## 1. Paths, frozen protocol, and run switches

The base model and Hugging Face revision match Phase 2. Hyperparameters and the
negative mode are hashed into the training manifest; the official-test gate
rejects a final model trained with a configuration different from the one chosen
on validation.

`NEGATIVE_MODE="in_batch"` is the recommended Phase 3A baseline. With MNRL, the
other positive documents in a batch already act as negatives, so explicit random
negatives are not required. `random` and `hard` are optional Phase 3B modes.


In [ ]:
COLAB_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/DuplicateDetection")
LOCAL_ROOT = Path(r"D:\Github\bughub")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

DATA_ROOT = COLAB_ROOT if COLAB_ROOT.exists() else LOCAL_ROOT
OUTPUT_ROOT = DATA_ROOT / "phase3_minilm_finetune_outputs"
CACHE_ROOT = OUTPUT_ROOT / "embedding_cache"
PAIR_ROOT = OUTPUT_ROOT / "training_pairs"
DEV_MODEL_ROOT = OUTPUT_ROOT / "dev_model"
FINAL_MODEL_ROOT = OUTPUT_ROOT / "final_model"
PHASE2_OUTPUT_ROOT = DATA_ROOT / "phase2_embedding_outputs"
for path in (OUTPUT_ROOT, CACHE_ROOT, PAIR_ROOT):
    path.mkdir(parents=True, exist_ok=True)

PROJECTS = {
    "Firefox": {
        "main": DATA_ROOT / "Firefox" / "mozilla_firefox.csv",
        "train": DATA_ROOT / "Firefox" / "train.csv",
        "test": DATA_ROOT / "Firefox" / "test.csv",
    },
    "MozillaCore": {
        "main": DATA_ROOT / "MozillaCore" / "mozilla_core.csv",
        "train": DATA_ROOT / "MozillaCore" / "train.csv",
        "test": DATA_ROOT / "MozillaCore" / "test.csv",
    },
}

VALIDATION_FRACTION_OF_OFFICIAL_TRAIN = 0.15
TOP_KS = (1, 5, 10)
BASE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BASE_MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"

# Compatibility globals used by the Phase 2 exact evaluator copied below.
EMBEDDING_MODEL_NAME = BASE_MODEL_NAME
EMBEDDING_MODEL_REVISION = BASE_MODEL_REVISION
EMBEDDING_ENCODING_MODE = "symmetric"
QUERY_PREFIX = ""
DOCUMENT_PREFIX = ""
DESCRIPTION_CHAR_LIMIT = 4_000
EMBEDDING_TEXT_VERSION = f"title-newline-description-head-{DESCRIPTION_CHAR_LIMIT}"
ENCODE_BATCH_SIZE = 128
EVAL_QUERY_BATCH_SIZE = 64
EVAL_DEVICE = None
TRUNCATION_AUDIT_SAMPLE = 2_000
TRUNCATION_WARNING_THRESHOLD_PCT = 20.0
FORCE_REBUILD_EMBEDDINGS = False

# Training-data policy.
ONE_PAIR_PER_DUPLICATE_CLUSTER = True
NEGATIVE_MODE = "in_batch"  # "in_batch", "random", or "hard"
HARD_NEGATIVE_SEARCH_DEPTH = 50
HARD_NEGATIVE_ABSOLUTE_MARGIN = 0.05

# Frozen Phase 3A training recipe.
LOSS_NAME = "cached_mnrl"
TRAIN_EPOCHS = 2.0
TRAIN_BATCH_SIZE = 128
LOSS_MINI_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0
USE_FP16 = True
REQUIRE_GPU_FOR_TRAINING = True
FORCE_RETRAIN = False

# Pre-registered comparison rule against pretrained MiniLM.
SELECTION_AGGREGATION = "macro_project_average"
PRIMARY_SELECTION_METRIC = "hit@10"
SELECTION_TIE_BREAKERS = ("mrr@10", "recall@10")

# Safe defaults: construct/audit pairs, but do not train or open test.
BUILD_DEV_PAIRS = True
RUN_DEV_TRAINING = False
RUN_DEV_VALIDATION = False
RUN_FINAL_RETRAIN = False
RUN_OFFICIAL_TEST = False
MAX_VALIDATION_QUERIES = None
MAX_TEST_QUERIES = None

for project, paths in PROJECTS.items():
    for kind, path in paths.items():
        assert path.exists(), f"Missing {project}/{kind}: {path}"
assert re.fullmatch(r"[0-9a-f]{40}", BASE_MODEL_REVISION)
assert NEGATIVE_MODE in {"in_batch", "random", "hard"}
assert LOSS_NAME == "cached_mnrl"
assert ONE_PAIR_PER_DUPLICATE_CLUSTER, (
    "MNRL baseline requires one pair per known duplicate cluster to prevent "
    "cluster members becoming false in-batch negatives."
)

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Base model:", BASE_MODEL_NAME)
print("Pinned revision:", BASE_MODEL_REVISION)
print("Negative mode:", NEGATIVE_MODE)
print("Build/train/validate/final/test:", BUILD_DEV_PAIRS, RUN_DEV_TRAINING,
      RUN_DEV_VALIDATION, RUN_FINAL_RETRAIN, RUN_OFFICIAL_TEST)


## 2. Load and normalize issues and official relevance files

`Duplicate` is a semicolon-separated multi-relevant set and is never reduced to
one target. Phase 2 keeps Phase 1's text normalization for identical data gates.
For the encoder, `embedding_text` contains the title once followed by the head of
the description; this avoids the Phase 1 TF-IDF title repetition and makes the
model's truncation policy explicit.


In [ ]:
def normalize_id(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return re.sub(r"\.0$", "", text)


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " <URL> ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_duplicate_set(value):
    if pd.isna(value):
        return frozenset()
    text = str(value).strip()
    if not text or text.upper() == "NULL":
        return frozenset()
    return frozenset(
        normalized
        for item in text.split(";")
        if (normalized := normalize_id(item))
    )


def load_project(project, paths):
    issues = pd.read_csv(paths["main"], low_memory=False)
    issues = issues.rename(columns={
        "Issue_id": "issue_id",
        "Priority": "priority",
        "Component": "component",
        "Duplicated_issue": "duplicate_of",
        "Title": "title",
        "Description": "description",
        "Resolution": "resolution",
        "Created_time": "created_at",
    })
    required = {
        "issue_id", "title", "description", "created_at",
        "duplicate_of", "component", "resolution",
    }
    assert required.issubset(issues.columns), required - set(issues.columns)

    issues["issue_id"] = issues["issue_id"].map(normalize_id)
    issues["created_at"] = pd.to_datetime(issues["created_at"], errors="coerce", utc=True)
    issues["title_clean"] = issues["title"].map(normalize_text)
    issues["description_clean"] = issues["description"].map(normalize_text)

    # Keep the Phase 1 field for identical non-empty-text filtering.
    issues["text"] = (
        issues["title_clean"] + " . " + issues["title_clean"]
        + " . " + issues["description_clean"]
    ).str.strip()

    description_head = issues["description_clean"].str.slice(0, DESCRIPTION_CHAR_LIMIT)
    issues["embedding_text"] = (
        issues["title_clean"] + "\n" + description_head
    ).str.strip()
    issues["description_char_truncated"] = (
        issues["description_clean"].str.len() > DESCRIPTION_CHAR_LIMIT
    )
    issues["project"] = project

    assert issues["issue_id"].notna().all()
    assert not issues["issue_id"].duplicated().any()
    assert issues["created_at"].notna().all()
    issues = issues.sort_values(["created_at", "issue_id"]).reset_index(drop=True)
    issues["position"] = np.arange(len(issues), dtype=np.int64)

    split_frames = {}
    for split in ("train", "test"):
        labels = pd.read_csv(paths[split], dtype=str)
        assert {"Issue_id", "Duplicate"}.issubset(labels.columns)
        labels = labels.rename(columns={"Issue_id": "issue_id", "Duplicate": "duplicate_raw"})
        labels["issue_id"] = labels["issue_id"].map(normalize_id)
        labels["relevant_ids"] = labels["duplicate_raw"].map(parse_duplicate_set)
        assert labels["issue_id"].notna().all()
        assert not labels["issue_id"].duplicated().any()
        split_frames[split] = labels[["issue_id", "relevant_ids"]].copy()

    all_ids = set(issues["issue_id"])
    train_ids = set(split_frames["train"]["issue_id"])
    test_ids = set(split_frames["test"]["issue_id"])
    assert train_ids.isdisjoint(test_ids), f"{project}: train/test ID overlap"
    assert train_ids | test_ids == all_ids, f"{project}: split does not cover main CSV"

    id_to_time = issues.set_index("issue_id")["created_at"].to_dict()
    id_to_text = issues.set_index("issue_id")["text"].to_dict()
    known = set(id_to_time)

    audit_rows = []
    for split, labels in split_frames.items():
        total_links = sum(len(x) for x in labels["relevant_ids"])
        missing = sum(
            target not in known
            for targets in labels["relevant_ids"]
            for target in targets
        )
        future = same = 0
        for query_id, targets in labels[["issue_id", "relevant_ids"]].itertuples(index=False):
            query_time = id_to_time[query_id]
            for target in targets:
                if target not in id_to_time:
                    continue
                future += id_to_time[target] > query_time
                same += id_to_time[target] == query_time
        audit_rows.append({
            "project": project,
            "split": split,
            "rows": len(labels),
            "labeled_queries": int((labels["relevant_ids"].map(len) > 0).sum()),
            "relevance_links": total_links,
            "missing_targets": missing,
            "future_targets": future,
            "same_time_targets": same,
            "description_char_truncated_pct": round(
                100 * issues["description_char_truncated"].mean(), 2
            ),
        })

    return {
        "project": project,
        "issues": issues,
        "train_labels": split_frames["train"],
        "test_labels": split_frames["test"],
        "id_to_time": id_to_time,
        "id_to_text": id_to_text,
        "audit": pd.DataFrame(audit_rows),
    }


datasets = {
    project: load_project(project, paths)
    for project, paths in PROJECTS.items()
}

data_audit = pd.concat([data["audit"] for data in datasets.values()], ignore_index=True)
display(data_audit)


## 3. Data-gate assertions

The official test files are suitable for rolling temporal retrieval only when
all labeled targets exist and none is from the future. Training label files may
contain future cluster members; validation construction filters those targets
at each query timestamp.


In [ ]:
dataset_summary = []
for project, data in datasets.items():
    issues = data["issues"]
    test_audit = data["audit"].query("split == 'test'").iloc[0]
    assert test_audit["missing_targets"] == 0
    assert test_audit["future_targets"] == 0
    assert not issues["issue_id"].duplicated().any()
    dataset_summary.append({
        "project": project,
        "issues": len(issues),
        "empty_text": int(issues["text"].str.len().eq(0).sum()),
        "created_min": issues["created_at"].min(),
        "created_max": issues["created_at"].max(),
        "official_train": len(data["train_labels"]),
        "official_test": len(data["test_labels"]),
        "test_labeled_queries": int(test_audit["labeled_queries"]),
        "test_relevance_links": int(test_audit["relevance_links"]),
    })

dataset_summary = pd.DataFrame(dataset_summary)
display(dataset_summary)
print("DATA GATE: PASS")


# Split-level leakage guard. Validation may use only official-train candidates.
# The current BugHub files are chronological, but the assertion makes that
# assumption executable instead of relying on the README or filename.
split_chronology_rows = []
for project, data in datasets.items():
    issues = data["issues"]
    official_train_ids = set(data["train_labels"]["issue_id"])
    official_test_ids = set(data["test_labels"]["issue_id"])
    train_times = issues.loc[
        issues["issue_id"].isin(official_train_ids), "created_at"
    ]
    test_times = issues.loc[
        issues["issue_id"].isin(official_test_ids), "created_at"
    ]
    train_max = train_times.max()
    test_min = test_times.min()
    assert train_max < test_min, (
        f"{project}: official split is not strictly chronological: "
        f"train_max={train_max}, test_min={test_min}. "
        "Keep the explicit allowed-candidate mask enabled."
    )
    data["official_train_ids"] = official_train_ids
    data["official_test_ids"] = official_test_ids
    split_chronology_rows.append({
        "project": project,
        "train_min": train_times.min(),
        "train_max": train_max,
        "test_min": test_min,
        "test_max": test_times.max(),
        "strictly_chronological": True,
    })

split_chronology = pd.DataFrame(split_chronology_rows)
display(split_chronology)
print("SPLIT CHRONOLOGY GATE: PASS")


## 4. Construct validation and test query sets

For every query, relevant reports are filtered to IDs that:

- exist in the same project corpus;
- have usable text;
- were created strictly before the query.

Queries with no remaining historical relevant report are not evaluable and are
reported separately. This prevents future-target leakage.


In [ ]:
def filter_temporal_queries(data, labels, allowed_query_ids=None):
    id_to_time = data["id_to_time"]
    id_to_text = data["id_to_text"]
    records = []
    audit = {
        "labeled_queries": 0,
        "empty_query_text": 0,
        "no_historical_relevant": 0,
        "future_or_same_relevant_removed": 0,
        "missing_or_empty_relevant_removed": 0,
    }
    allowed_query_ids = None if allowed_query_ids is None else set(allowed_query_ids)

    for query_id, raw_relevant in labels[["issue_id", "relevant_ids"]].itertuples(index=False):
        if allowed_query_ids is not None and query_id not in allowed_query_ids:
            continue
        if not raw_relevant:
            continue
        audit["labeled_queries"] += 1
        if not id_to_text.get(query_id, ""):
            audit["empty_query_text"] += 1
            continue

        query_time = id_to_time[query_id]
        relevant = set()
        for target in raw_relevant:
            if target not in id_to_time or not id_to_text.get(target, ""):
                audit["missing_or_empty_relevant_removed"] += 1
                continue
            if id_to_time[target] >= query_time:
                audit["future_or_same_relevant_removed"] += 1
                continue
            relevant.add(target)

        if not relevant:
            audit["no_historical_relevant"] += 1
            continue
        records.append({
            "query_id": query_id,
            "query_time": query_time,
            "relevant_ids": frozenset(relevant),
            "num_relevant": len(relevant),
        })

    query_frame = pd.DataFrame(records).sort_values(["query_time", "query_id"]).reset_index(drop=True)
    audit["evaluable_queries"] = len(query_frame)
    audit["evaluable_relevance_links"] = int(query_frame["num_relevant"].sum()) if len(query_frame) else 0
    return query_frame, audit


query_sets = {}
query_audits = []
for project, data in datasets.items():
    issues = data["issues"]
    train_labels = data["train_labels"]
    official_train_ids = set(train_labels["issue_id"])
    train_issues = issues[issues["issue_id"].isin(official_train_ids)].sort_values(["created_at", "issue_id"])

    validation_size = math.ceil(len(train_issues) * VALIDATION_FRACTION_OF_OFFICIAL_TRAIN)
    validation_ids = set(train_issues.tail(validation_size)["issue_id"])
    dev_fit_ids = set(train_issues.iloc[:-validation_size]["issue_id"])

    validation_queries, validation_audit = filter_temporal_queries(
        data, train_labels, validation_ids
    )
    test_queries, test_audit = filter_temporal_queries(data, data["test_labels"])

    assert all(
        data["id_to_time"][target] < row.query_time
        for row in validation_queries.itertuples()
        for target in row.relevant_ids
    )
    assert all(
        data["id_to_time"][target] < row.query_time
        for row in test_queries.itertuples()
        for target in row.relevant_ids
    )

    query_sets[project] = {
        "dev_fit_ids": dev_fit_ids,
        "official_train_ids": official_train_ids,
        "validation": validation_queries,
        "test": test_queries,
    }
    for split, audit in (("validation", validation_audit), ("test", test_audit)):
        query_audits.append({"project": project, "split": split, **audit})

query_audits = pd.DataFrame(query_audits)
display(query_audits)
print("TEMPORAL GROUND TRUTH: PASS")


## 5. Exact temporal retrieval engine

This is the hardened Phase 2 evaluator: normalized vectors, exact batched inner
product, strictly older candidates, and an explicit split candidate mask. It is
used unchanged for pretrained and fine-tuned MiniLM.


In [ ]:
def ranking_metrics(retrieved_ids, relevant_ids, ks=TOP_KS):
    relevant = set(relevant_ids)
    result = {}
    for k in ks:
        hits = len(set(retrieved_ids[:k]) & relevant)
        result[f"hit@{k}"] = float(hits > 0)
        result[f"recall@{k}"] = hits / len(relevant)

    first_rank = next(
        (rank for rank, issue_id in enumerate(retrieved_ids[:10], start=1)
         if issue_id in relevant),
        None,
    )
    result["mrr@10"] = 0.0 if first_rank is None else 1.0 / first_rank

    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, issue_id in enumerate(retrieved_ids[:10], start=1)
        if issue_id in relevant
    )
    ideal_hits = min(len(relevant), 10)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    result["ndcg@10"] = dcg / idcg if idcg else 0.0
    return result


def top_k_indices(scores, k):
    k = min(k, len(scores))
    if k == 0:
        return np.array([], dtype=np.int64)
    if k == len(scores):
        candidates = np.arange(len(scores))
    else:
        candidates = np.argpartition(-scores, k - 1)[:k]
    # Deterministic: score descending, then chronological corpus position.
    order = np.lexsort((candidates, -scores[candidates]))
    return candidates[order]


def deterministic_query_sample(queries, max_queries):
    if max_queries is None or len(queries) <= max_queries:
        return queries.reset_index(drop=True)
    selected = np.linspace(0, len(queries) - 1, max_queries, dtype=int)
    return queries.iloc[selected].drop_duplicates("query_id").reset_index(drop=True)


def embedding_fingerprint(issues, model_name, model_revision, cache_role):
    digest = hashlib.sha256()
    cache_settings = {
        "model_name": model_name,
        "model_revision": model_revision,
        "encoding_mode": EMBEDDING_ENCODING_MODE,
        "cache_role": cache_role,
        "query_prefix": QUERY_PREFIX,
        "document_prefix": DOCUMENT_PREFIX,
        "embedding_text_version": EMBEDDING_TEXT_VERSION,
        "sentence_transformers_version": sentence_transformers.__version__,
        "torch_version": torch.__version__,
    }
    digest.update(json.dumps(cache_settings, sort_keys=True).encode("utf-8"))
    for issue_id, text in issues[["issue_id", "embedding_text"]].itertuples(index=False):
        digest.update(issue_id.encode("utf-8"))
        digest.update(b"\0")
        digest.update(text.encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()[:20]


def safe_model_slug(model_name):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", model_name).strip("_")


class SentenceTransformerBackend:
    """Role-aware adapter; symmetric MiniLM remains the zero-cost default."""

    def __init__(self, model_name, model_revision):
        if not model_name or not model_revision:
            raise ValueError("Both model_name and pinned model_revision are required")
        self.model_name = model_name
        self.model_revision = model_revision
        self.model = SentenceTransformer(model_name, revision=model_revision)

    def prepare_texts(self, texts, role):
        if role not in {"query", "document"}:
            raise ValueError(f"Unknown embedding role: {role}")
        prefix = QUERY_PREFIX if role == "query" else DOCUMENT_PREFIX
        return [f"{prefix}{text}" for text in texts] if prefix else texts

    def encode_prepared(self, prepared_texts, role):
        method = self.model.encode
        if EMBEDDING_ENCODING_MODE == "query_document":
            role_method = "encode_query" if role == "query" else "encode_document"
            method = getattr(self.model, role_method, self.model.encode)
        return method(
            prepared_texts,
            batch_size=ENCODE_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32, copy=False)


_embedding_backend = None


def get_embedding_backend():
    global _embedding_backend
    if _embedding_backend is None:
        _embedding_backend = SentenceTransformerBackend(
            EMBEDDING_MODEL_NAME, EMBEDDING_MODEL_REVISION
        )
    return _embedding_backend


def audit_model_truncation(model, texts, sample_size=2_000, batch_size=128):
    if not texts:
        return {"token_audit_sample": 0, "model_max_tokens": None,
                "token_truncated_pct_estimate": None}
    sample_size = min(sample_size, len(texts))
    positions = np.linspace(0, len(texts) - 1, sample_size, dtype=int)
    sample = [texts[position] for position in positions]
    lengths = []
    for start in range(0, len(sample), batch_size):
        encoded = model.tokenizer(
            sample[start:start + batch_size],
            add_special_tokens=True,
            padding=False,
            truncation=False,
        )
        lengths.extend(len(ids) for ids in encoded["input_ids"])
    max_tokens = int(model.max_seq_length)
    audit = {
        "token_audit_sample": len(lengths),
        "model_max_tokens": max_tokens,
        "token_length_median": float(np.median(lengths)),
        "token_length_p95": float(np.percentile(lengths, 95)),
        "truncation_warning_threshold_pct": TRUNCATION_WARNING_THRESHOLD_PCT,
        "token_truncated_pct_estimate": round(
            100 * np.mean(np.asarray(lengths) > max_tokens), 2
        ),
    }
    if audit["token_truncated_pct_estimate"] > TRUNCATION_WARNING_THRESHOLD_PCT:
        print(
            "WARNING: substantial embedding input truncation: "
            f"{audit['token_truncated_pct_estimate']:.2f}% > "
            f"{TRUNCATION_WARNING_THRESHOLD_PCT:.2f}% threshold"
        )
    return audit


def build_or_load_embeddings(data, role, force_rebuild=False):
    if role not in {"query", "document"}:
        raise ValueError(f"Unknown embedding role: {role}")
    issues = data["issues"]
    cache_role = "shared" if EMBEDDING_ENCODING_MODE == "symmetric" else role
    fingerprint = embedding_fingerprint(
        issues, EMBEDDING_MODEL_NAME, EMBEDDING_MODEL_REVISION, cache_role
    )
    stem = (
        f"{data['project'].lower()}__{safe_model_slug(EMBEDDING_MODEL_NAME)}"
        f"__{cache_role}__{fingerprint}"
    )
    matrix_path = CACHE_ROOT / f"{stem}.npy"
    metadata_path = CACHE_ROOT / f"{stem}.json"

    if matrix_path.exists() and metadata_path.exists() and not force_rebuild:
        embeddings = np.load(matrix_path)
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        assert embeddings.shape[0] == len(issues)
        assert embeddings.dtype == np.float32
        metadata["cache_hit"] = True
        print(f"CACHE HIT: {data['project']} -> {matrix_path.name}")
        return embeddings, metadata

    print(f"ENCODING: {data['project']} / {cache_role} ({len(issues):,} reports)")
    backend = get_embedding_backend()
    encode_role = "document" if cache_role == "shared" else role
    texts = backend.prepare_texts(issues["embedding_text"].tolist(), encode_role)
    truncation_metadata = audit_model_truncation(
        backend.model, texts, TRUNCATION_AUDIT_SAMPLE, ENCODE_BATCH_SIZE
    )
    started = time.time()
    embeddings = backend.encode_prepared(texts, encode_role)

    assert embeddings.ndim == 2 and embeddings.shape[0] == len(issues)
    norm_sample = np.linalg.norm(embeddings[:min(2_000, len(embeddings))], axis=1)
    nonzero = norm_sample > 0
    assert np.allclose(norm_sample[nonzero], 1.0, atol=1e-4)

    metadata = {
        "project": data["project"],
        "representation": "embedding",
        "model_name": EMBEDDING_MODEL_NAME,
        "model_revision": EMBEDDING_MODEL_REVISION,
        "encoding_mode": EMBEDDING_ENCODING_MODE,
        "cache_role": cache_role,
        "query_prefix": QUERY_PREFIX,
        "document_prefix": DOCUMENT_PREFIX,
        "embedding_text_version": EMBEDDING_TEXT_VERSION,
        "fingerprint": fingerprint,
        "reports": len(issues),
        "embedding_dim": int(embeddings.shape[1]),
        "dtype": str(embeddings.dtype),
        "encode_seconds": round(time.time() - started, 2),
        "sentence_transformers_version": sentence_transformers.__version__,
        "torch_version": torch.__version__,
        "cache_hit": False,
        **truncation_metadata,
    }
    np.save(matrix_path, embeddings)
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"CACHE SAVED: {matrix_path}")
    display(pd.DataFrame([metadata]))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return embeddings, metadata


def evaluate_dense_rolling(
    data,
    queries,
    document_embeddings,
    query_embeddings,
    model_name,
    model_revision,
    allowed_candidate_ids,
    max_queries=None,
    query_batch_size=64,
    device=None,
):
    issues = data["issues"]
    assert document_embeddings.shape[0] == len(issues)
    assert query_embeddings.shape[0] == len(issues)
    eval_queries = deterministic_query_sample(queries, max_queries)
    assert len(eval_queries) > 0

    issue_ids = issues["issue_id"].to_numpy()
    created_ns = issues["created_at"].astype("int64").to_numpy()
    id_to_position = dict(zip(issue_ids, issues["position"]))
    query_positions = np.asarray(
        [id_to_position[query_id] for query_id in eval_queries["query_id"]],
        dtype=np.int64,
    )
    query_times_ns = eval_queries["query_time"].astype("int64").to_numpy()
    candidate_ends = np.searchsorted(created_ns, query_times_ns, side="left")
    assert np.all(candidate_ends > 0)
    assert np.all(candidate_ends <= query_positions)

    allowed_candidate_ids = set(allowed_candidate_ids)
    allowed_mask = np.fromiter(
        (issue_id in allowed_candidate_ids for issue_id in issue_ids),
        dtype=bool,
        count=len(issue_ids),
    )
    assert allowed_mask.any()

    resolved_device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    corpus = torch.from_numpy(np.asarray(document_embeddings)).to(resolved_device)
    shared_query_document_matrix = query_embeddings is document_embeddings
    query_matrix = (
        corpus
        if shared_query_document_matrix
        else torch.from_numpy(np.asarray(query_embeddings)).to(resolved_device)
    )
    rows = []
    started = time.time()
    max_k = max(TOP_KS)

    with torch.inference_mode():
        for start in range(0, len(eval_queries), query_batch_size):
            stop = min(start + query_batch_size, len(eval_queries))
            batch_positions = query_positions[start:stop]
            batch_candidate_ends = candidate_ends[start:stop]
            batch_corpus_end = int(batch_candidate_ends.max())

            query_tensor = query_matrix[
                torch.as_tensor(batch_positions, dtype=torch.long, device=resolved_device)
            ]
            score_batch = (
                query_tensor @ corpus[:batch_corpus_end].T
            ).float().cpu().numpy()

            for offset, row in enumerate(
                eval_queries.iloc[start:stop].itertuples(index=False)
            ):
                candidate_end = int(batch_candidate_ends[offset])
                local_scores = score_batch[offset, :candidate_end]
                local_allowed = allowed_mask[:candidate_end]
                if local_allowed.all():
                    local_top = top_k_indices(local_scores, max_k)
                else:
                    legal_positions = np.flatnonzero(local_allowed)
                    assert len(legal_positions) > 0
                    legal_top = top_k_indices(local_scores[legal_positions], max_k)
                    local_top = legal_positions[legal_top]
                retrieved = issue_ids[local_top].tolist()

                assert row.query_id not in retrieved
                assert set(retrieved).issubset(allowed_candidate_ids)
                assert all(data["id_to_time"][item] < row.query_time for item in retrieved)
                metrics = ranking_metrics(retrieved, row.relevant_ids)
                rows.append({
                    "project": data["project"],
                    "representation": "embedding",
                    "model_name": model_name,
                    "model_revision": model_revision,
                    "encoding_mode": EMBEDDING_ENCODING_MODE,
                    "query_id": row.query_id,
                    "query_time": row.query_time,
                    "num_candidates": int(local_allowed.sum()),
                    "num_relevant": len(row.relevant_ids),
                    "retrieved_ids": ";".join(retrieved),
                    **metrics,
                })

            completed = stop
            if completed % 256 == 0 or completed == len(eval_queries):
                print(f"  {data['project']} / embedding: {completed}/{len(eval_queries)}")
            del query_tensor, score_batch

    elapsed = time.time() - started
    results = pd.DataFrame(rows)
    metric_columns = [
        *(f"hit@{k}" for k in TOP_KS),
        *(f"recall@{k}" for k in TOP_KS),
        "mrr@10", "ndcg@10",
    ]
    summary = {
        "project": data["project"],
        "representation": "embedding",
        "model_name": model_name,
        "model_revision": model_revision,
        "encoding_mode": EMBEDDING_ENCODING_MODE,
        "embedding_text_version": EMBEDDING_TEXT_VERSION,
        "queries": len(results),
        "median_candidates": int(results["num_candidates"].median()),
        "retrieval_seconds": round(elapsed, 2),
        "evaluation_device": resolved_device,
        "query_batch_size": query_batch_size,
        **{column: results[column].mean() for column in metric_columns},
    }

    if not shared_query_document_matrix:
        del query_matrix
    del corpus
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results, summary


# Metric and ranking unit tests.
metric_check = ranking_metrics(["x", "b", "y"], {"a", "b"}, ks=(1, 2, 3))
assert metric_check["hit@1"] == 0
assert metric_check["hit@2"] == 1
assert metric_check["recall@2"] == 0.5
assert metric_check["mrr@10"] == 0.5
assert top_k_indices(np.array([0.1, 0.9, 0.9], dtype=np.float32), 2).tolist() == [1, 2]
print("METRIC + TOP-K TESTS: PASS")


## 6. Build leakage-safe positive pairs

Only `train.csv` relevance links are read. For development, both sides must be in
`dev_fit_ids`; validation and official-test IDs are rejected by assertions. Each
duplicate relation is oriented as:

```text
newer report (anchor/query) -> older duplicate report (positive/document)
```

Known duplicate links form connected components. Phase 3A samples one observed
historical edge per component, so another positive from the same component cannot
become a false in-batch negative under MNRL.


In [ ]:
class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, item):
        self.parent.setdefault(item, item)
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])
        return self.parent[item]

    def union(self, left, right):
        left_root, right_root = self.find(left), self.find(right)
        if left_root == right_root:
            return
        # Stable root makes cluster IDs reproducible.
        small, large = sorted((left_root, right_root))
        self.parent[large] = small


def build_positive_pairs(data, allowed_ids, scope_name, seed=RANDOM_SEED):
    allowed_ids = set(allowed_ids)
    issues = data["issues"]
    id_to_time = data["id_to_time"]
    id_to_text = issues.set_index("issue_id")["embedding_text"].to_dict()
    union_find = UnionFind()
    observed_pairs = set()
    audit = {
        "project": data["project"],
        "scope": scope_name,
        "raw_train_links": 0,
        "outside_scope_removed": 0,
        "same_time_removed": 0,
        "empty_text_removed": 0,
    }

    # Never inspect data["test_labels"] here.
    for query_id, targets in data["train_labels"][["issue_id", "relevant_ids"]].itertuples(index=False):
        for target_id in targets:
            audit["raw_train_links"] += 1
            if query_id not in allowed_ids or target_id not in allowed_ids:
                audit["outside_scope_removed"] += 1
                continue
            if not id_to_text.get(query_id, "") or not id_to_text.get(target_id, ""):
                audit["empty_text_removed"] += 1
                continue
            query_time, target_time = id_to_time[query_id], id_to_time[target_id]
            if query_time == target_time:
                audit["same_time_removed"] += 1
                continue

            anchor_id, positive_id = (
                (query_id, target_id)
                if query_time > target_time
                else (target_id, query_id)
            )
            observed_pairs.add((anchor_id, positive_id))
            union_find.union(query_id, target_id)

    rows = []
    for anchor_id, positive_id in sorted(observed_pairs):
        root = union_find.find(anchor_id)
        rows.append({
            "project": data["project"],
            "scope": scope_name,
            "cluster_id": f"{data['project']}::{root}",
            "anchor_id": anchor_id,
            "positive_id": positive_id,
            "anchor_time": id_to_time[anchor_id],
            "positive_time": id_to_time[positive_id],
            "anchor": id_to_text[anchor_id],
            "positive": id_to_text[positive_id],
        })
    pairs = pd.DataFrame(rows)
    assert len(pairs) > 0
    assert (pairs["positive_time"] < pairs["anchor_time"]).all()
    assert set(pairs["anchor_id"]).issubset(allowed_ids)
    assert set(pairs["positive_id"]).issubset(allowed_ids)

    cluster_members = {}
    for issue_id in union_find.parent:
        cluster_id = f"{data['project']}::{union_find.find(issue_id)}"
        cluster_members.setdefault(cluster_id, set()).add(issue_id)

    if ONE_PAIR_PER_DUPLICATE_CLUSTER:
        # One deterministic random observed edge per connected component.
        pairs = (
            pairs.groupby("cluster_id", group_keys=False, sort=True)
            .sample(n=1, random_state=seed)
            .sort_values(["anchor_time", "anchor_id", "positive_id"])
            .reset_index(drop=True)
        )
        assert pairs["cluster_id"].is_unique

    audit.update({
        "unique_historical_edges": len(observed_pairs),
        "duplicate_clusters": len(cluster_members),
        "selected_training_pairs": len(pairs),
    })
    return pairs, cluster_members, audit


def build_scope_pairs(scope_name):
    pair_frames, audits, component_maps = [], [], {}
    for project, data in datasets.items():
        if scope_name == "dev_fit":
            allowed_ids = query_sets[project]["dev_fit_ids"]
        elif scope_name == "official_train":
            allowed_ids = query_sets[project]["official_train_ids"]
        else:
            raise ValueError(f"Unknown pair scope: {scope_name}")
        pairs, members, audit = build_positive_pairs(data, allowed_ids, scope_name)
        pair_frames.append(pairs)
        audits.append(audit)
        component_maps[project] = members

    combined = pd.concat(pair_frames, ignore_index=True)
    assert combined["cluster_id"].is_unique
    return combined, component_maps, pd.DataFrame(audits)


dev_pairs = pd.DataFrame()
dev_component_maps = {}
dev_pair_audit = pd.DataFrame()
if BUILD_DEV_PAIRS or RUN_DEV_TRAINING:
    dev_pairs, dev_component_maps, dev_pair_audit = build_scope_pairs("dev_fit")
    for project in PROJECTS:
        validation_ids = set(query_sets[project]["validation"]["query_id"])
        test_ids = set(datasets[project]["test_labels"]["issue_id"])
        project_pairs = dev_pairs.query("project == @project")
        used_ids = set(project_pairs["anchor_id"]) | set(project_pairs["positive_id"])
        assert used_ids.isdisjoint(validation_ids)
        assert used_ids.isdisjoint(test_ids)
    display(dev_pair_audit)
    display(dev_pairs.head())
    dev_pairs.to_csv(PAIR_ROOT / "dev_fit_positive_pairs.csv", index=False)
    print("DEV PAIR LEAKAGE GATE: PASS")
else:
    print("DEV PAIR BUILD SKIPPED")


## 7. Optional explicit negatives

- `in_batch` (default): no explicit negative column; MNRL uses the other batch
  positives as negatives.
- `random`: one historical, same-project report outside the anchor's known
  duplicate component.
- `hard`: the highest-ranked legal non-duplicate under frozen pretrained MiniLM,
  subject to an absolute margin below the positive score. Mining is restricted to
  the same training scope and therefore cannot inspect validation/test candidates.

Hard negatives are an optional Phase 3B experiment. Do not switch modes after
looking at official-test results.


In [ ]:
def allowed_ids_for_scope(project, scope_name):
    if scope_name == "dev_fit":
        return set(query_sets[project]["dev_fit_ids"])
    if scope_name == "official_train":
        return set(query_sets[project]["official_train_ids"])
    raise ValueError(scope_name)


def add_random_negatives(pairs, component_maps, scope_name, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    output = pairs.copy()
    negatives, negative_ids = [], []
    for row in output.itertuples(index=False):
        data = datasets[row.project]
        issues = data["issues"]
        allowed = allowed_ids_for_scope(row.project, scope_name)
        forbidden = component_maps[row.project][row.cluster_id]
        legal = issues[
            issues["issue_id"].isin(allowed)
            & (issues["created_at"] < row.anchor_time)
            & ~issues["issue_id"].isin(forbidden)
            & issues["embedding_text"].ne("")
        ]
        assert len(legal) > 0
        chosen = legal.iloc[int(rng.integers(0, len(legal)))]
        negative_ids.append(chosen["issue_id"])
        negatives.append(chosen["embedding_text"])
    output["negative_id"] = negative_ids
    output["negative"] = negatives
    return output


def add_hard_negatives(pairs, component_maps, scope_name):
    """Exact pretrained-MiniLM mining; intentionally disabled by default."""
    output_frames = []
    for project, project_pairs in pairs.groupby("project", sort=True):
        data = datasets[project]
        issues = data["issues"]
        allowed = allowed_ids_for_scope(project, scope_name)
        base_embeddings, _ = build_or_load_embeddings(
            data=data, role="document", force_rebuild=FORCE_REBUILD_EMBEDDINGS
        )
        ids = issues["issue_id"].to_numpy()
        times_ns = issues["created_at"].astype("int64").to_numpy()
        id_to_pos = dict(zip(ids, issues["position"]))
        allowed_mask = np.fromiter(
            (issue_id in allowed for issue_id in ids), dtype=bool, count=len(ids)
        )
        device = EVAL_DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
        matrix = torch.from_numpy(np.asarray(base_embeddings)).to(device)
        rows = []
        with torch.inference_mode():
            for row in project_pairs.itertuples(index=False):
                anchor_pos = id_to_pos[row.anchor_id]
                positive_pos = id_to_pos[row.positive_id]
                candidate_end = int(np.searchsorted(
                    times_ns, int(row.anchor_time.value), side="left"
                ))
                scores = (matrix[anchor_pos] @ matrix[:candidate_end].T).float().cpu().numpy()
                positive_score = float(
                    (matrix[anchor_pos] @ matrix[positive_pos]).float().cpu()
                )
                forbidden = component_maps[project][row.cluster_id]
                legal_positions = np.flatnonzero(allowed_mask[:candidate_end])
                forbidden_positions = np.asarray(
                    [id_to_pos[item] for item in forbidden if item in id_to_pos],
                    dtype=np.int64,
                )
                if len(forbidden_positions):
                    legal_positions = legal_positions[
                        ~np.isin(legal_positions, forbidden_positions)
                    ]
                assert len(legal_positions) > 0
                ranked_local = top_k_indices(
                    scores[legal_positions],
                    min(HARD_NEGATIVE_SEARCH_DEPTH, len(legal_positions)),
                )
                ranked_positions = legal_positions[ranked_local]
                eligible = [
                    position for position in ranked_positions
                    if scores[position] <= positive_score - HARD_NEGATIVE_ABSOLUTE_MARGIN
                ]
                chosen_pos = eligible[0] if eligible else int(ranked_positions[-1])
                record = row._asdict()
                record["negative_id"] = ids[chosen_pos]
                record["negative"] = issues.iloc[chosen_pos]["embedding_text"]
                record["positive_base_score"] = positive_score
                record["negative_base_score"] = float(scores[chosen_pos])
                rows.append(record)
        output_frames.append(pd.DataFrame(rows))
        del matrix
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pd.concat(output_frames, ignore_index=True)


def prepare_training_pairs(base_pairs, component_maps, scope_name):
    if NEGATIVE_MODE == "in_batch":
        prepared = base_pairs.copy()
    elif NEGATIVE_MODE == "random":
        prepared = add_random_negatives(base_pairs, component_maps, scope_name)
    elif NEGATIVE_MODE == "hard":
        prepared = add_hard_negatives(base_pairs, component_maps, scope_name)
    else:
        raise ValueError(NEGATIVE_MODE)

    required = {"anchor", "positive"}
    if NEGATIVE_MODE != "in_batch":
        required.add("negative")
    assert required.issubset(prepared.columns)
    assert prepared["cluster_id"].is_unique
    return prepared


dev_training_pairs = pd.DataFrame()
if RUN_DEV_TRAINING:
    assert len(dev_pairs) > 0
    dev_training_pairs = prepare_training_pairs(
        dev_pairs, dev_component_maps, "dev_fit"
    )
    dev_training_pairs.to_csv(
        PAIR_ROOT / f"dev_fit_{NEGATIVE_MODE}_training_pairs.csv", index=False
    )


## 8. Development fine-tuning on `dev_fit` only

The trainer always initializes from the pinned pretrained MiniLM. It uses
`CachedMultipleNegativesRankingLoss`, a large logical batch, a small GradCache
mini-batch, and `BatchSamplers.NO_DUPLICATES`. No evaluator is called during
training; the fixed two-epoch recipe is evaluated once by the rolling validation
section.


In [ ]:
def stable_pair_hash(pair_frame):
    columns = ["project", "cluster_id", "anchor_id", "positive_id"]
    if "negative_id" in pair_frame:
        columns.append("negative_id")
    payload = pair_frame[columns].sort_values(columns).to_csv(index=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def training_config():
    return {
        "base_model_name": BASE_MODEL_NAME,
        "base_model_revision": BASE_MODEL_REVISION,
        "embedding_text_version": EMBEDDING_TEXT_VERSION,
        "one_pair_per_duplicate_cluster": ONE_PAIR_PER_DUPLICATE_CLUSTER,
        "negative_mode": NEGATIVE_MODE,
        "hard_negative_search_depth": HARD_NEGATIVE_SEARCH_DEPTH,
        "hard_negative_absolute_margin": HARD_NEGATIVE_ABSOLUTE_MARGIN,
        "loss_name": LOSS_NAME,
        "epochs": TRAIN_EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "loss_mini_batch_size": LOSS_MINI_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "max_grad_norm": MAX_GRAD_NORM,
        "random_seed": RANDOM_SEED,
    }


def training_config_hash():
    payload = json.dumps(training_config(), sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def train_biencoder(pair_frame, model_root, scope_name):
    final_path = model_root / "final"
    manifest_path = model_root / "training_manifest.json"
    if final_path.exists() and manifest_path.exists() and not FORCE_RETRAIN:
        saved = json.loads(manifest_path.read_text(encoding="utf-8"))
        assert saved["training_config_hash"] == training_config_hash()
        assert saved["scope"] == scope_name
        assert saved["pair_hash"] == stable_pair_hash(pair_frame)
        print("TRAINING CACHE HIT:", final_path)
        return SentenceTransformer(str(final_path)), saved

    if REQUIRE_GPU_FOR_TRAINING:
        assert torch.cuda.is_available(), "Enable a Colab GPU before training."
    model = SentenceTransformer(BASE_MODEL_NAME, revision=BASE_MODEL_REVISION)
    dataset_columns = ["anchor", "positive"]
    if NEGATIVE_MODE != "in_batch":
        dataset_columns.append("negative")
    train_dataset = Dataset.from_pandas(
        pair_frame[dataset_columns], preserve_index=False
    )
    loss = losses.CachedMultipleNegativesRankingLoss(
        model=model,
        mini_batch_size=LOSS_MINI_BATCH_SIZE,
    )
    args = SentenceTransformerTrainingArguments(
        output_dir=str(model_root / "checkpoints"),
        overwrite_output_dir=FORCE_RETRAIN,
        num_train_epochs=TRAIN_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        max_grad_norm=MAX_GRAD_NORM,
        fp16=bool(USE_FP16 and torch.cuda.is_available()),
        bf16=False,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        eval_strategy="no",
        save_strategy="epoch",
        save_total_limit=2,
        logging_strategy="steps",
        logging_steps=25,
        report_to="none",
        seed=RANDOM_SEED,
        data_seed=RANDOM_SEED,
        dataloader_num_workers=2,
        run_name=f"phase3-{scope_name}-{NEGATIVE_MODE}",
    )
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        loss=loss,
    )
    trainer.train()
    final_path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(final_path))

    for project, project_pairs in pair_frame.groupby("project"):
        used_ids = set(project_pairs["anchor_id"]) | set(project_pairs["positive_id"])
        if "negative_id" in project_pairs:
            used_ids |= set(project_pairs["negative_id"])
        assert used_ids.issubset(allowed_ids_for_scope(project, scope_name))
        assert used_ids.isdisjoint(
            set(datasets[project]["test_labels"]["issue_id"])
        )

    manifest = {
        "scope": scope_name,
        "pair_rows": len(pair_frame),
        "pair_hash": stable_pair_hash(pair_frame),
        "training_config": training_config(),
        "training_config_hash": training_config_hash(),
        "used_development_validation_pairs": scope_name == "official_train",
        "used_official_test_pairs": False,
        "sentence_transformers_version": sentence_transformers.__version__,
        "torch_version": torch.__version__,
        "model_path": str(final_path),
    }
    model_root.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return model, manifest


dev_model = None
dev_training_manifest = None
if RUN_DEV_TRAINING:
    dev_model, dev_training_manifest = train_biencoder(
        dev_training_pairs, DEV_MODEL_ROOT, "dev_fit"
    )
else:
    print("DEV TRAINING LOCKED: set RUN_DEV_TRAINING=True on a GPU.")


## 9. Full development validation — pretrained vs fine-tuned

Both models encode the same Phase 2 text and use the same query IDs, ground truth,
official-train candidate mask, strictly older history, and exact Top-K evaluator.
The section writes per-query paired outcomes and a frozen selection artifact.

The fine-tuned model must win the pre-registered macro-project comparison before
the final retrain/test gates can open.


In [ ]:
def checkpoint_fingerprint(model_path):
    digest = hashlib.sha256()
    model_path = Path(model_path)
    included = []
    for path in sorted(model_path.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix not in {".safetensors", ".bin", ".json", ".txt"}:
            continue
        relative = path.relative_to(model_path).as_posix()
        included.append(relative)
        digest.update(relative.encode("utf-8"))
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
    assert included, f"No checkpoint files found under {model_path}"
    return digest.hexdigest()


def encode_model_for_project(model, data, model_tag, model_fingerprint):
    issues = data["issues"]
    text_digest = hashlib.sha256()
    text_digest.update(model_fingerprint.encode("utf-8"))
    text_digest.update(EMBEDDING_TEXT_VERSION.encode("utf-8"))
    for issue_id, text in issues[["issue_id", "embedding_text"]].itertuples(index=False):
        text_digest.update(issue_id.encode("utf-8"))
        text_digest.update(b"\0")
        text_digest.update(text.encode("utf-8"))
    key = text_digest.hexdigest()[:20]
    path = CACHE_ROOT / f"{data['project'].lower()}__{model_tag}__{key}.npy"
    if path.exists() and not FORCE_REBUILD_EMBEDDINGS:
        matrix = np.load(path)
        assert matrix.shape[0] == len(issues)
        print("EVAL CACHE HIT:", path.name)
        return matrix
    matrix = model.encode(
        issues["embedding_text"].tolist(),
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32, copy=False)
    np.save(path, matrix)
    return matrix


def metric_tuple(summary_row):
    return tuple(float(summary_row[metric]) for metric in (
        PRIMARY_SELECTION_METRIC, *SELECTION_TIE_BREAKERS
    ))


dev_validation_summary = pd.DataFrame()
paired_validation_summary = pd.DataFrame()
phase3_selection = None

if RUN_DEV_VALIDATION:
    dev_manifest_path = DEV_MODEL_ROOT / "training_manifest.json"
    dev_checkpoint = DEV_MODEL_ROOT / "final"
    assert dev_manifest_path.exists() and dev_checkpoint.exists(), (
        "Train the dev model before validation."
    )
    dev_training_manifest = json.loads(dev_manifest_path.read_text(encoding="utf-8"))
    assert dev_training_manifest["scope"] == "dev_fit"
    assert dev_training_manifest["training_config_hash"] == training_config_hash()
    dev_model = SentenceTransformer(str(dev_checkpoint))
    base_model = SentenceTransformer(BASE_MODEL_NAME, revision=BASE_MODEL_REVISION)
    tuned_fingerprint = checkpoint_fingerprint(dev_checkpoint)

    summary_rows, paired_rows = [], []
    for project, data in datasets.items():
        print(f"\n=== PHASE 3 VALIDATION: {project} ===")
        base_embeddings = encode_model_for_project(
            base_model, data, "pretrained", BASE_MODEL_REVISION
        )
        tuned_embeddings = encode_model_for_project(
            dev_model, data, "dev_finetuned", tuned_fingerprint
        )
        model_results = {}
        for label, matrix, model_name, revision in (
            ("pretrained", base_embeddings, BASE_MODEL_NAME, BASE_MODEL_REVISION),
            ("fine_tuned", tuned_embeddings, str(dev_checkpoint), tuned_fingerprint),
        ):
            results, summary = evaluate_dense_rolling(
                data=data,
                queries=query_sets[project]["validation"],
                document_embeddings=matrix,
                query_embeddings=matrix,
                model_name=model_name,
                model_revision=revision,
                allowed_candidate_ids=data["official_train_ids"],
                max_queries=MAX_VALIDATION_QUERIES,
                query_batch_size=EVAL_QUERY_BATCH_SIZE,
                device=EVAL_DEVICE,
            )
            results["system"] = label
            summary["system"] = label
            model_results[label] = results
            summary_rows.append(summary)
            results.to_csv(
                OUTPUT_ROOT / f"{project.lower()}_validation_{label}_queries.csv",
                index=False,
            )

        paired = model_results["pretrained"][["query_id", "hit@10"]].merge(
            model_results["fine_tuned"][["query_id", "hit@10"]],
            on="query_id", suffixes=("_pretrained", "_fine_tuned"), validate="one_to_one",
        )
        paired["project"] = project
        paired["outcome"] = np.select(
            [
                (paired["hit@10_pretrained"] == 0) & (paired["hit@10_fine_tuned"] == 1),
                (paired["hit@10_pretrained"] == 1) & (paired["hit@10_fine_tuned"] == 0),
                (paired["hit@10_pretrained"] == 1) & (paired["hit@10_fine_tuned"] == 1),
            ],
            ["fine_tuned_win", "pretrained_win", "both_correct"],
            default="both_fail",
        )
        paired_rows.append(paired)

    project_summary = pd.DataFrame(summary_rows)
    metrics = [
        *(f"hit@{k}" for k in TOP_KS), *(f"recall@{k}" for k in TOP_KS),
        "mrr@10", "ndcg@10",
    ]
    macro_rows = []
    for system, group in project_summary.groupby("system", sort=True):
        macro_rows.append({
            "project": "MACRO_PROJECT_AVERAGE",
            "system": system,
            "queries": int(group["queries"].sum()),
            **{metric: group[metric].mean() for metric in metrics},
        })
    dev_validation_summary = pd.concat(
        [project_summary, pd.DataFrame(macro_rows)], ignore_index=True
    )
    display(dev_validation_summary)
    dev_validation_summary.to_csv(OUTPUT_ROOT / "dev_validation_summary.csv", index=False)

    paired_validation = pd.concat(paired_rows, ignore_index=True)
    paired_validation.to_csv(OUTPUT_ROOT / "paired_validation_queries.csv", index=False)
    paired_validation_summary = (
        paired_validation.groupby(["project", "outcome"]).size()
        .rename("queries").reset_index()
    )
    display(paired_validation_summary)

    macro = dev_validation_summary.query("project == 'MACRO_PROJECT_AVERAGE'").set_index("system")
    pretrained_score = metric_tuple(macro.loc["pretrained"])
    fine_tuned_score = metric_tuple(macro.loc["fine_tuned"])
    winner = "fine_tuned" if fine_tuned_score > pretrained_score else "pretrained"
    phase3_selection = {
        "aggregation": SELECTION_AGGREGATION,
        "primary_metric": PRIMARY_SELECTION_METRIC,
        "tie_breakers": list(SELECTION_TIE_BREAKERS),
        "winner": winner,
        "pretrained_score_tuple": list(pretrained_score),
        "fine_tuned_score_tuple": list(fine_tuned_score),
        "training_config_hash": training_config_hash(),
        "dev_pair_hash": dev_training_manifest["pair_hash"],
        "full_validation": MAX_VALIDATION_QUERIES is None,
        "official_test_open_allowed": (
            winner == "fine_tuned" and MAX_VALIDATION_QUERIES is None
        ),
    }
    (OUTPUT_ROOT / "phase3_model_selection.json").write_text(
        json.dumps(phase3_selection, indent=2), encoding="utf-8"
    )
    print(json.dumps(phase3_selection, indent=2))
else:
    print("DEV VALIDATION LOCKED: train dev model, then set RUN_DEV_VALIDATION=True.")


## 10. Final retrain on complete official train

This gate opens only when the fine-tuned system wins a complete validation run.
It rebuilds pairs using all official-train IDs and initializes again from the
pinned pretrained MiniLM. Validation pairs may now be included because model and
hyperparameters are already frozen. Official-test IDs remain forbidden.


In [ ]:
final_training_pairs = pd.DataFrame()
final_training_manifest = None

if RUN_FINAL_RETRAIN:
    selection_path = OUTPUT_ROOT / "phase3_model_selection.json"
    assert selection_path.exists(), "Missing Phase 3 validation selection artifact."
    selected = json.loads(selection_path.read_text(encoding="utf-8"))
    assert selected["official_test_open_allowed"] is True
    assert selected["winner"] == "fine_tuned"
    assert selected["training_config_hash"] == training_config_hash()

    final_pairs, final_component_maps, final_pair_audit = build_scope_pairs("official_train")
    display(final_pair_audit)
    final_training_pairs = prepare_training_pairs(
        final_pairs, final_component_maps, "official_train"
    )
    final_training_pairs.to_csv(
        PAIR_ROOT / f"official_train_{NEGATIVE_MODE}_training_pairs.csv", index=False
    )
    final_model, final_training_manifest = train_biencoder(
        final_training_pairs, FINAL_MODEL_ROOT, "official_train"
    )
else:
    print("FINAL RETRAIN LOCKED until fine-tuned MiniLM wins full validation.")


## 11. Official test — run once

The official test requires the frozen validation selection and a final model
retrained on full official train with exactly the selected configuration. Test
labels are used only to score retrieval, never to create pairs or negatives.


In [ ]:
test_summary_with_macro = pd.DataFrame()
test_results = {}

if RUN_OFFICIAL_TEST:
    assert MAX_TEST_QUERIES is None, "Publishable official test must be unsampled."
    selection = json.loads(
        (OUTPUT_ROOT / "phase3_model_selection.json").read_text(encoding="utf-8")
    )
    final_manifest = json.loads(
        (FINAL_MODEL_ROOT / "training_manifest.json").read_text(encoding="utf-8")
    )
    assert selection["official_test_open_allowed"] is True
    assert selection["winner"] == "fine_tuned"
    assert selection["training_config_hash"] == training_config_hash()
    assert final_manifest["scope"] == "official_train"
    assert final_manifest["training_config_hash"] == training_config_hash()
    assert final_manifest["used_official_test_pairs"] is False

    final_checkpoint = FINAL_MODEL_ROOT / "final"
    assert final_checkpoint.exists()
    final_model = SentenceTransformer(str(final_checkpoint))
    final_fingerprint = checkpoint_fingerprint(final_checkpoint)
    summaries = []
    for project, data in datasets.items():
        print(f"\n=== PHASE 3 OFFICIAL TEST: {project} ===")
        matrix = encode_model_for_project(
            final_model, data, "final_finetuned", final_fingerprint
        )
        results, summary = evaluate_dense_rolling(
            data=data,
            queries=query_sets[project]["test"],
            document_embeddings=matrix,
            query_embeddings=matrix,
            model_name=str(final_checkpoint),
            model_revision=final_fingerprint,
            allowed_candidate_ids=(
                data["official_train_ids"] | data["official_test_ids"]
            ),
            max_queries=MAX_TEST_QUERIES,
            query_batch_size=EVAL_QUERY_BATCH_SIZE,
            device=EVAL_DEVICE,
        )
        test_results[project] = results
        summaries.append(summary)
        results.to_csv(
            OUTPUT_ROOT / f"{project.lower()}_official_test_queries.csv", index=False
        )

    project_test_summary = pd.DataFrame(summaries)
    metrics = [
        *(f"hit@{k}" for k in TOP_KS), *(f"recall@{k}" for k in TOP_KS),
        "mrr@10", "ndcg@10",
    ]
    macro = {
        "project": "MACRO_PROJECT_AVERAGE",
        "system": "fine_tuned",
        "queries": int(project_test_summary["queries"].sum()),
        **{metric: project_test_summary[metric].mean() for metric in metrics},
    }
    test_summary_with_macro = pd.concat(
        [project_test_summary, pd.DataFrame([macro])], ignore_index=True
    )
    display(test_summary_with_macro)
    test_summary_with_macro.to_csv(OUTPUT_ROOT / "official_test_summary.csv", index=False)
else:
    print("OFFICIAL TEST SEALED.")


## 12. Run manifest and final leakage assertions

In [ ]:
for project, results in test_results.items():
    data = datasets[project]
    assert results["query_id"].is_unique
    for row in results.itertuples(index=False):
        query_time = data["id_to_time"][row.query_id]
        assert all(
            data["id_to_time"][retrieved_id] < query_time
            for retrieved_id in row.retrieved_ids.split(";") if retrieved_id
        )

run_manifest = {
    "phase": 3,
    "objective": "fine-tune MiniLM bi-encoder for temporal duplicate retrieval",
    "base_model_name": BASE_MODEL_NAME,
    "base_model_revision": BASE_MODEL_REVISION,
    "training_config": training_config(),
    "training_config_hash": training_config_hash(),
    "selection_rule": {
        "aggregation": SELECTION_AGGREGATION,
        "primary_metric": PRIMARY_SELECTION_METRIC,
        "tie_breakers": list(SELECTION_TIE_BREAKERS),
    },
    "run_switches": {
        "build_dev_pairs": BUILD_DEV_PAIRS,
        "run_dev_training": RUN_DEV_TRAINING,
        "run_dev_validation": RUN_DEV_VALIDATION,
        "run_final_retrain": RUN_FINAL_RETRAIN,
        "run_official_test": RUN_OFFICIAL_TEST,
    },
    "full_publishable_run": (
        RUN_OFFICIAL_TEST
        and MAX_VALIDATION_QUERIES is None
        and MAX_TEST_QUERIES is None
    ),
    "environment": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": torch.__version__,
        "sentence_transformers": sentence_transformers.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    "projects": {
        project: {
            "issues": len(data["issues"]),
            "dev_fit_ids": len(query_sets[project]["dev_fit_ids"]),
            "validation_queries": len(query_sets[project]["validation"]),
            "official_train_ids": len(query_sets[project]["official_train_ids"]),
            "official_test_queries": len(query_sets[project]["test"]),
        }
        for project, data in datasets.items()
    },
}
(OUTPUT_ROOT / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2), encoding="utf-8"
)
print(json.dumps(run_manifest, indent=2))
print("PHASE 3 LEAKAGE ASSERTIONS: PASS")


## Reporting and next decision

Report Phase 3A as a fixed-recipe fine-tuning experiment:

- training scope: `dev_fit` only during development;
- positive policy: one historical observed edge per known duplicate component;
- loss: Cached Multiple Negatives Ranking Loss;
- negatives: in-batch by default;
- primary decision metric: macro-project Hit@10;
- tie-breakers: macro MRR@10, then Recall@10;
- paired outcome counts on identical validation query IDs.

If pretrained MiniLM wins validation, do not open official test and do not claim
that fine-tuning works. Hard-negative mining is then a new validation-only Phase
3B experiment. If fine-tuned MiniLM wins, freeze the recipe, retrain from the
original base model on complete official train, and execute official test once.
